# LIBRERIAS Y CARGA DEL MODELO

# QWEN V2.5

In [ ]:
# Isntalación de dependencias necesarias para ejecutar el proyecto
!pip install fastapi uvicorn python-multipart -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!pip install bitsandbytes -q
!pip install "transformers>=4.49.0" -q
!pip install "qwen-vl-utils>=0.0.10" -q
print("✓ Listo. Reinicia el kernel y ejecuta desde la siguiente celda.")

In [ ]:
# Librerias para el servidor FastAPI y el modelo Qwen2.5-VL.
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, HTMLResponse
import uvicorn, threading, io, subprocess, time, re
from PIL import Image
import json
from pathlib import Path

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

import cv2
import numpy as np
import json

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

In [ ]:
DEVICE

In [ ]:
# Para modelo de 4 bits (Cuantizacion)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# Cargar del modelo y procesador
qwen_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256  * 28 * 28,
    max_pixels=1280 * 28 * 28,
)

In [ ]:
# Carga del modelo sin cuantizar
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    #quantization_config=bnb_config,   ← comentado = cuantización desactivada
    device_map="auto",
)
qwen_model.eval()

In [ ]:
# Info del vram utilizado
if torch.cuda.is_available():
    usado = torch.cuda.memory_allocated() / 1e9
    libre = torch.cuda.get_device_properties(0).total_memory / 1e9 - usado
    print(f"✓ Qwen2.5-VL-7B cargado | VRAM usada: {usado:.1f} GB | libre: {libre:.1f} GB")

In [ ]:
# PREPROCESAMIENTO DE IMÁGENES
def _deskew(img):
    gris   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bordes = cv2.Canny(gris, 50, 150, apertureSize=3)
    lineas = cv2.HoughLines(bordes, 1, np.pi/180, threshold=120)
    if lineas is None: return img
    angulos = [np.degrees(t)-90 for _,(r,t) in enumerate(lineas[:,0])
               if np.pi/4 < t < 3*np.pi/4]
    if not angulos: return img
    ang = float(np.median(angulos))
    if abs(ang) < 0.3: return img
    h,w = img.shape[:2]
    M   = cv2.getRotationMatrix2D((w//2,h//2), ang, 1.0)
    return cv2.warpAffine(img, M, (w,h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)

def _clahe(img):
    lab    = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L,A,B  = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8))
    return cv2.cvtColor(cv2.merge([clahe.apply(L),A,B]),
                        cv2.COLOR_LAB2BGR)

def _sharp(img):
    k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
    return cv2.addWeighted(img, 0.4, cv2.filter2D(img,-1,k), 0.6, 0)

In [ ]:
# Función que aplica el preprocesamiento a una imagen PIL
def preprocesar(pil_img):
    if pil_img.mode != "RGB":
        pil_img = pil_img.convert("RGB")
    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    img = _deskew(img)
    img = _clahe(img)
    img = _sharp(img)
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [ ]:
PROMPT = """Eres un experto en documentos civiles peruanos del RENIEC.
Esta imagen es un acta de nacimiento peruana con texto impreso y manuscrito.
Extrae y transcribe EXACTAMENTE todos los campos visibles.

Responde SOLO con este JSON (sin texto extra, sin bloques de código):
{
  "numero_acta": "",
  "fecha_nacimiento": "",
  "hora_nacimiento": "",
  "departamento": "",
  "provincia": "",
  "distrito": "",
  "sexo": "",
  "titular_nombres": "",
  "titular_apellidos": "",
  "padre_nombres": "",
  "padre_apellidos": "",
  "padre_nacionalidad": "",
  "padre_dni": "",
  "padre_ocupacion": "",
  "padre_departamento": "",
  "padre_provincia": "",
  "padre_distrito": "",
  "madre_nombres": "",
  "madre_apellidos": "",
  "madre_nacionalidad": "",
  "madre_dni": "",
  "madre_ocupacion": "",
  "madre_departamento": "",
  "madre_provincia": "",
  "madre_distrito": "",
  "fecha_registro": "",
  "oficina_registral": "",
  "encargado_nombre": "",
  "encargado_cargo": "",
  "encargado_dni": "",
  "encargado_departamento": "",
  "encargado_provincia": "",
  "encargado_distrito": "",
  "encargado_observaciones": ""
}
Si un campo es ilegible escribe [ilegible].
Si no existe en el documento escribe [no presente]."""

In [ ]:
# Función principal que recibe una imagen PIL y devuelve un diccionario con los datos extraídos
def transcribir_acta(imagen_pil: Image.Image) -> dict:

    # Preprocesamiento de la imagen
    imagen = preprocesar(imagen_pil)

    # Creación de los mensajes para el modelo
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": imagen},
            {"type": "text",  "text": PROMPT},
        ],
    }]

    # PREPARACIÓN DE LOS INPUTS PARA EL MODELO
    texto_entrada = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Procesamiento de imagenes
    img_inputs, vid_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[texto_entrada],
        images=img_inputs,
        videos=vid_inputs,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    # Generacion de respuestas
    with torch.no_grad():
        out_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            repetition_penalty=1.05,
        )

    # Para obtener solo la parte generada, se recortan los tokens del prompt.
    trimmed = [o[len(i):] for i,o in zip(inputs.input_ids, out_ids)]
    respuesta = qwen_processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    # Se  parsea la respuesta como JSON. Si falla, se devuelve la respuesta cruda en un campo específico.
    try:
        if "```" in respuesta:
            respuesta = respuesta.split("```")[1]
            if respuesta.startswith("json"):
                respuesta = respuesta[4:]
        campos = json.loads(respuesta.strip())
    except json.JSONDecodeError:
        campos = {"respuesta_cruda": respuesta}

    torch.cuda.empty_cache()
    return campos

# QWEN v3.0

In [ ]:
# LIBRERIAS PARA CLOUDFLARED
!pip install fastapi uvicorn python-multipart -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
# LIBRERIAS NECESARIAS PARA QWEN 7B
!pip install bitsandbytes -q
#!pip install "transformers>=4.49.0" -q
!pip install git+https://github.com/huggingface/transformers -q
!pip install "qwen-vl-utils>=0.0.10" -q
print("✓ Listo. Reinicia el kernel y ejecuta desde la siguiente celda.")

In [1]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, HTMLResponse
import uvicorn, threading, io, subprocess, time, re
from PIL import Image
import json
from pathlib import Path
#----------------------------------------------------------------------
# Celda 2 — Cargar Qwen3-VL-7B
import torch
#from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
#--------------------------------------------------------------------------------
import cv2
import numpy as np
import json

In [2]:
# DESCARGAR QWEN
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
#MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# 7B en 4-bit = ~5-6 GB VRAM — entra en T4 sin problema
# Quantización
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Cargando {MODEL_ID}...")

qwen_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256  * 28 * 28,
    max_pixels=1280 * 28 * 28,
)

qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    #torch_dtype=torch.float16,
    torch_dtype="auto",
    #quantization_config=bnb_config,
    device_map="auto",
)

qwen_model.eval()

if torch.cuda.is_available():
    usado = torch.cuda.memory_allocated() / 1e9
    libre = torch.cuda.get_device_properties(0).total_memory / 1e9 - usado
    print(f"✓ Qwen2.5-VL-7B cargado | VRAM usada: {usado:.1f} GB | libre: {libre:.1f} GB")

Cargando Qwen/Qwen3-VL-8B-Instruct...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

✓ Qwen2.5-VL-7B cargado | VRAM usada: 7.8 GB | libre: 7.8 GB


In [4]:
# Celda 3 — Nuevo pipeline
# ── Preprocesamiento ─────────────────────────────
def _deskew(img):
    gris   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bordes = cv2.Canny(gris, 50, 150, apertureSize=3)
    lineas = cv2.HoughLines(bordes, 1, np.pi/180, threshold=120)
    if lineas is None: return img
    angulos = [np.degrees(t)-90 for _,(r,t) in enumerate(lineas[:,0])
               if np.pi/4 < t < 3*np.pi/4]
    if not angulos: return img
    ang = float(np.median(angulos))
    if abs(ang) < 0.3: return img
    h,w = img.shape[:2]
    M   = cv2.getRotationMatrix2D((w//2,h//2), ang, 1.0)
    return cv2.warpAffine(img, M, (w,h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)

def _clahe(img):
    lab    = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L,A,B  = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8))
    return cv2.cvtColor(cv2.merge([clahe.apply(L),A,B]),
                        cv2.COLOR_LAB2BGR)

def _sharp(img):
    k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
    return cv2.addWeighted(img, 0.4, cv2.filter2D(img,-1,k), 0.6, 0)

def preprocesar(pil_img):
    if pil_img.mode != "RGB":
        pil_img = pil_img.convert("RGB")
    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    img = _deskew(img)
    img = _clahe(img)
    img = _sharp(img)
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

# ── Prompt estructurado para actas RENIEC ─────────────────────
# Qwen2.5-VL extrae todos los campos directamente
PROMPT = """Eres un experto en documentos civiles peruanos del RENIEC.
Esta imagen es un acta de nacimiento peruana con texto impreso y manuscrito.
Extrae y transcribe EXACTAMENTE todos los campos visibles.

Responde SOLO con este JSON (sin texto extra, sin bloques de código):
{
  "numero_acta": "",
  "fecha_nacimiento": "",
  "hora_nacimiento": "",
  "departamento": "",
  "provincia": "",
  "distrito": "",
  "sexo": "",
  "titular_nombres": "",
  "titular_apellidos": "",
  "padre_nombres": "",
  "padre_apellidos": "",
  "padre_nacionalidad": "",
  "padre_dni": "",
  "padre_ocupacion": "",
  "padre_departamento": "",
  "padre_provincia": "",
  "padre_distrito": "",
  "madre_nombres": "",
  "madre_apellidos": "",
  "madre_nacionalidad": "",
  "madre_dni": "",
  "madre_ocupacion": "",
  "madre_departamento": "",
  "madre_provincia": "",
  "madre_distrito": "",
  "fecha_registro": "",
  "oficina_registral": "",
  "encargado_nombre": "",
  "encargado_cargo": "",
  "encargado_dni": "",
  "encargado_departamento": "",
  "encargado_provincia": "",
  "encargado_distrito": "",
  "encargado_observaciones": ""
}
Si un campo es ilegible escribe [ilegible].
Si no existe en el documento escribe [no presente]."""


def transcribir_acta(imagen_pil: Image.Image) -> dict:
    """
    Pipeline completo usando solo Qwen2.5-VL-7B.
    Preprocesa la imagen y extrae todos los campos del acta en JSON.
    """
    # 1. Preprocesar
    imagen = preprocesar(imagen_pil)

    # 2. Qwen2.5-VL transcribe la imagen completa
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": imagen},
            {"type": "text",  "text": PROMPT},
        ],
    }]
    
    '''
    texto_entrada        = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    img_inputs, vid_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[texto_entrada],
        images=img_inputs,
        videos=vid_inputs,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)
    '''
    inputs = qwen_processor.apply_chat_template(
        messages, 
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True, 
        return_tensors="pt",
    ).to(DEVICE)
    

    with torch.no_grad():
        out_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=512,
            #do_sample=False,
            #repetition_penalty=1.05,
            do_sample=True, 
            temperature=0.7, 
            top_p=0.8, 
            top_k=20, 
            repetition_penalty=1.0

        )

    trimmed = [o[len(i):] for i,o in zip(inputs.input_ids, out_ids)]
    respuesta = qwen_processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    # 3. Parsear el JSON de la respuesta
    try:
        # Qwen a veces agrega ```json ... ``` — limpiar si lo hace
        if "```" in respuesta:
            respuesta = respuesta.split("```")[1]
            if respuesta.startswith("json"):
                respuesta = respuesta[4:]
        campos = json.loads(respuesta.strip())
    except json.JSONDecodeError:
        # Si no devolvió JSON válido, devolver la respuesta cruda
        campos = {"respuesta_cruda": respuesta}

    torch.cuda.empty_cache()
    return campos

# SERVIDOR

In [5]:
# Servidor 
import subprocess, time, re, threading, io, base64
subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True)
time.sleep(2)

from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, asyncio

CARPETA_JSON = Path("/kaggle/working/resultados_actas")
CARPETA_JSON.mkdir(exist_ok=True)

app = FastAPI()
app.add_middleware(CORSMiddleware,
                   allow_origins=["*"],
                   allow_methods=["*"],
                   allow_headers=["*"])

def img_a_b64(pil_img, max_w=900):
    w,h = pil_img.size
    if w > max_w:
        pil_img = pil_img.resize((max_w, int(h*max_w/w)), Image.LANCZOS)
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

@app.get("/ping")
def ping(): return {"status": "ok", "modelo": "Qwen2.5-VL-7B"}

@app.post("/transcribir")
async def transcribir(file: UploadFile = File(...)):
    imagen = Image.open(io.BytesIO(await file.read())).convert("RGB")
    campos = transcribir_acta(imagen)

        # ── Guardar JSON en disco ──────────────────────────────────
    # Nombre del archivo: timestamp para que no se sobreescriban
    from datetime import datetime
    timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_json = f"acta_{timestamp}.json"
    ruta_json   = CARPETA_JSON / nombre_json

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(campos, f, ensure_ascii=False, indent=2)

    print(f"[guardado] {ruta_json}")
    # ──────────────────────────────────────────────────────────

    # Devolver también la imagen preprocesada para mostrar en el front
    img_proc = preprocesar(imagen)

    return JSONResponse({
        "campos":  campos,
        "imagen":  img_a_b64(img_proc),
        "archivo_guardado": nombre_json,# imagen preprocesada sin boxes
    })

def _run():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    cfg = uvicorn.Config(app, host="0.0.0.0", port=7860,
                         log_level="error", loop="none")
    loop.run_until_complete(uvicorn.Server(cfg).serve())

threading.Thread(target=_run, daemon=True).start()
time.sleep(3)
print("✓ Servidor listo")

proc = subprocess.Popen(
    ["./cloudflared","tunnel","--url","http://localhost:7860"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)
for _ in range(30):
    linea = proc.stderr.readline().decode("utf-8", errors="ignore")
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", linea)
    if m:
        print(f"\n✅ URL    : {m.group()}")
        print(f"✅ Ping   : {m.group()}/ping")
        break
    time.sleep(1)

✓ Servidor listo

✅ URL    : https://laws-enjoyed-propecia-glenn.trycloudflare.com
✅ Ping   : https://laws-enjoyed-propecia-glenn.trycloudflare.com/ping
[guardado] /kaggle/working/resultados_actas/acta_20260611_135118.json
